## Natural Language Processing Spring 2026
---
# Model Architecture (Live Coding)

# Introduction

**MiniMind Step-by-Step Series** | Milestone 2 of 6

> **Live Coding Session** — Infrastructure is pre-filled.
> Implement the **core building blocks** in the `# TODO` sections.

**Learning Objectives:**
- Define model hyperparameters with MiniMindConfig
- Implement RMSNorm (Root Mean Square Layer Normalization)
- Implement Rotary Position Embeddings (RoPE)
- Build Grouped-Query Attention (GQA)
- Implement the SwiGLU Feed-Forward Network
- Understand Mixture of Experts (MOE) routing
- Assemble a Transformer Block with pre-norm residual connections
- Wire everything into a complete MiniMind causal LM and count parameters


In [1]:
# === Setup (recap from Notebook 1) ===
!pip install tokenizers torch transformers --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

PyTorch 2.10.0+cu128  CUDA: True


## Part A — Model Configuration

Before building layers, we define the model hyperparameters. MiniMind is a ~26M parameter transformer decoder with these key settings:

| Parameter | Value | Purpose |
|---|---|---|
| `hidden_size` | 768 | Dimension of hidden representations |
| `num_hidden_layers` | 8 | Number of transformer blocks |
| `num_attention_heads` | 8 | Number of query heads |
| `num_key_value_heads` | 4 | Fewer KV heads → Grouped-Query Attention |
| `head_dim` | 96 | Per-head dimension (hidden_size / num_attention_heads) |
| `intermediate_size` | 2048 | FFN inner dimension |
| `vocab_size` | 6400 | Our BPE vocabulary from Notebook 1 |
| `max_position_embeddings` | 32768 | Maximum sequence length |
| `rope_theta` | 1e6 | RoPE base frequency |

**Why these values?** MiniMind is designed to be small enough to train on a single GPU while retaining the architectural patterns of production LLMs (LLaMA, Gemma, Qwen). Here is how the key parameters interact:

- **`hidden_size = 768`** sets the model's "bandwidth" — every token is represented as a 768-dimensional vector throughout the network. Larger values capture richer representations but use more memory and compute. 768 is a common choice for small models (GPT-2 Small also uses 768).
- **`head_dim = hidden_size / num_attention_heads = 96`** — each attention head operates on a 96-dimensional subspace. Heads work in parallel, each attending to different relationship patterns. The full hidden state is split evenly across heads.
- **`intermediate_size = 2048 ≈ 2.67 × hidden_size`** — the feed-forward network expands to a higher dimension to increase expressiveness before projecting back. Production models often use a 4× ratio; MiniMind uses a smaller ratio to keep parameter count low.
- **`num_key_value_heads = 4`** (half of `num_attention_heads = 8`) — this is Grouped-Query Attention (explained in Part D). Each KV head serves 2 query heads, cutting the KV cache in half during inference.
- **`rope_theta = 1e6`** — a high base frequency for RoPE (explained in Part C), which helps the model generalize to longer sequences than it was trained on.
- **MOE parameters** (`num_experts`, `num_experts_per_tok`, `moe_intermediate_size`) — these configure an optional Mixture of Experts layer (explained in Part E½). By default MOE is disabled (`use_moe=False`).

In [2]:
# === Model Configuration ===
@dataclass
class MiniMindConfig:
    hidden_size: int = 768
    num_hidden_layers: int = 8
    num_attention_heads: int = 8
    num_key_value_heads: int = 4       # GQA: fewer KV heads than Q heads
    head_dim: int = 96                  # hidden_size // num_attention_heads
    vocab_size: int = 6400
    intermediate_size: int = 2048
    max_position_embeddings: int = 32768
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1e6
    dropout: float = 0.0
    hidden_act: str = 'silu'
    flash_attn: bool = True
    bos_token_id: int = 1
    eos_token_id: int = 2
    use_moe: bool = False              # Enable Mixture of Experts
    num_experts: int = 4
    num_experts_per_tok: int = 1
    moe_intermediate_size: int = 2048
    norm_topk_prob: bool = True
    router_aux_loss_coef: float = 5e-4

config = MiniMindConfig()
print(f'Config: hidden_size={config.hidden_size}, layers={config.num_hidden_layers}')
print(f'  Q heads={config.num_attention_heads}, KV heads={config.num_key_value_heads}')
print(f'  head_dim={config.head_dim}, vocab_size={config.vocab_size}')
print(f'  intermediate_size={config.intermediate_size}')

Config: hidden_size=768, layers=8
  Q heads=8, KV heads=4
  head_dim=96, vocab_size=6400
  intermediate_size=2048


## Part B — RMSNorm

Layer normalization stabilizes training by normalizing activations. **RMSNorm** (Root Mean Square Normalization) is a simplified variant used in modern transformers like LLaMA and MiniMind:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \cdot \gamma$$

**Why normalize at all?** Without normalization, activations can grow or shrink as they pass through many layers, causing gradient explosion or vanishing. Normalization keeps activations in a stable range, which lets us use higher learning rates and train deeper networks.

**Step-by-step computation:**
1. **Square** each element: $x^2$
2. **Mean** across the last dimension (the hidden dimension): $\text{mean}(x^2)$
3. **Reciprocal square root**: $\frac{1}{\sqrt{\text{mean}(x^2) + \epsilon}}$ — this is the RMS normalization factor ($\epsilon$ prevents division by zero)
4. **Scale** each element by this factor: $x \cdot \text{rsqrt}(\text{mean}(x^2) + \epsilon)$
5. **Multiply** by the learnable weight $\gamma$: the network can adjust the scale per dimension

**RMSNorm vs LayerNorm:** Standard LayerNorm computes both the mean $\mu$ and variance $\sigma^2$, then normalizes as $(x - \mu) / \sigma$. RMSNorm **skips the mean subtraction** — it only rescales by the root-mean-square. The key insight from the [RMSNorm paper (Zhang & Sennrich, 2019)](https://arxiv.org/abs/1910.07467) is that the re-centering (mean subtraction) in LayerNorm is not essential for training stability — the re-scaling alone is sufficient. This makes RMSNorm faster (fewer operations, no mean computation) while performing comparably for transformers.

**Where is RMSNorm applied?** In MiniMind, RMSNorm appears in four places: (1) before the attention sublayer, (2) before the FFN sublayer (both are "pre-norm" — see Part F), (3) after the final transformer block, and (4) inside the attention module to normalize Q and K (QK normalization for training stability).

In [3]:
# === RMSNorm ===
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def norm(self, x):
        # ============================================================
        # TODO: Implement RMS normalization
        #
        # Formula: x * rsqrt(mean(x^2) + eps)
        #
        # Steps:
        #   1. Square x element-wise: x.pow(2)
        #   2. Mean over last dim, keep dims: .mean(-1, keepdim=True)
        #   3. Add eps for numerical stability
        #   4. Take reciprocal square root: torch.rsqrt(...)
        #   5. Multiply x by the result
        #
        # Hint: torch.rsqrt(a) == 1 / sqrt(a)
        # ============================================================


        x_sqrt = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * x_sqrt

    def forward(self, x):
        return (self.weight * self.norm(x.float())).type_as(x)

print('RMSNorm defined \u2713')


RMSNorm defined ✓


In [4]:
# === Test RMSNorm ===
rms = RMSNorm(dim=768)
x = torch.randn(2, 32, 768)
out = rms(x)

assert out.shape == x.shape, f'Shape mismatch: {out.shape}'
rms_vals = out.float().pow(2).mean(-1).sqrt()
print(f'Output shape: {out.shape}')
print(f'RMS values (should be ≈ 1): mean={rms_vals.mean():.4f}, std={rms_vals.std():.4f}')
print('RMSNorm test passed')

Output shape: torch.Size([2, 32, 768])
RMS values (should be ≈ 1): mean=1.0000, std=0.0000
RMSNorm test passed


## Part C — Rotary Position Embeddings (RoPE)

Transformers need to know token positions — without position information, "the cat sat on the mat" and "mat the on sat cat the" would produce the same attention patterns. **RoPE** encodes position by rotating query/key vectors in 2D subspaces.

**The core idea:** Split each head's $d$-dimensional vector into $d/2$ pairs of adjacent dimensions. Each pair forms a 2D plane, and we rotate it by an angle that depends on the token's position:

$$\theta_i = \text{base}^{-2i/d}$$

$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

where $m$ is the position index and $i$ is the dimension pair index.

**Why rotations work for relative position:** When computing attention scores $q' \cdot k'$, the rotation angles are $m\theta_i$ for the query (at position $m$) and $n\theta_i$ for the key (at position $n$). Because rotations compose, the dot product depends only on the **difference** $(m - n)\theta_i$, not on the absolute positions. This gives the model translation-invariant position awareness — the model learns to attend based on how far apart tokens are, regardless of where they appear in the sequence.

**Comparison with other position encodings:**
| Method | Type | Key Property |
|---|---|---|
| Sinusoidal (original Transformer) | Fixed, additive | Adds position vectors to embeddings. Fixed but not relative. |
| Learned absolute | Learned, additive | Learns a position vector per position. Can't generalize beyond training length. |
| **RoPE** | **Fixed, multiplicative** | **Rotates Q/K — naturally relative, extrapolates to longer sequences.** |
| ALiBi | Fixed, attention bias | Adds linear bias to attention scores. No modification to Q/K. |

**The frequency spectrum:** Each dimension pair rotates at a different frequency controlled by $\theta_i$. Low-indexed pairs rotate fast (capturing fine-grained, nearby positions), while high-indexed pairs rotate slowly (capturing long-range structure). The `rope_base` parameter (1e6 in MiniMind) controls how quickly frequencies decay — a larger base means slower rotation in higher dimensions, which helps the model generalize to sequences longer than those seen in training.

**Implementation detail:** We precompute $\cos(m\theta_i)$ and $\sin(m\theta_i)$ tables for all positions up to `max_position_embeddings`. During the forward pass, we slice these tables to the current sequence length and apply them to Q and K via element-wise multiplication (`rotate_half` rearranges dimensions to implement the 2×2 rotation matrix efficiently).

In [ ]:
# === Rotary Position Embeddings (RoPE) ===
def precompute_freqs_cis(dim, end=32768, rope_base=1e6):
    """Precompute cosine and sine tables for RoPE."""
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))
    # ============================================================
    # TODO: Build cosine and sine frequency tables
    #
    # Given per-dimension frequencies `freqs` (shape [dim//2]):
    #   1. Create position indices: t = torch.arange(end)
    #   2. Outer product: freqs = torch.outer(t, freqs).float()
    #      → shape (end, dim//2), each row = position × per-dim freq
    #   3. Duplicate across both halves (needed for rotate_half):
    #      freqs_cos = torch.cat([cos(freqs), cos(freqs)], dim=-1)
    #      freqs_sin = torch.cat([sin(freqs), sin(freqs)], dim=-1)
    #   4. Return freqs_cos, freqs_sin  (each shape: end × dim)
    # ============================================================

    # YOUR CODE HERE
    pass

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    """Apply rotary embeddings to query and key tensors."""
    def rotate_half(x):
        # Split x in half along last dim and swap/negate halves
        return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)
    # ============================================================
    # TODO: Apply the rotation to q and k
    #
    # Steps:
    #   1. Add a batch dimension to cos and sin:
    #      cos = cos.unsqueeze(unsqueeze_dim)
    #      sin = sin.unsqueeze(unsqueeze_dim)
    #   2. Rotate: q_embed = (q * cos) + (rotate_half(q) * sin)
    #              k_embed = (k * cos) + (rotate_half(k) * sin)
    #   3. Cast back to original dtype: .to(q.dtype) / .to(k.dtype)
    #   4. Return q_embed, k_embed
    # ============================================================

    # YOUR CODE HERE
    pass

print('RoPE functions defined \u2713')


In [ ]:
# === Test RoPE ===
HEAD_DIM = 96
SEQ_LEN = 32
freqs_cos, freqs_sin = precompute_freqs_cis(dim=HEAD_DIM, end=SEQ_LEN)

assert freqs_cos.shape == (SEQ_LEN, HEAD_DIM), f'freqs_cos shape: {freqs_cos.shape}'
assert freqs_sin.shape == (SEQ_LEN, HEAD_DIM), f'freqs_sin shape: {freqs_sin.shape}'

# Test rotation preserves dot-product relative structure
q = torch.randn(2, SEQ_LEN, 8, HEAD_DIM)
k = torch.randn(2, SEQ_LEN, 4, HEAD_DIM)
q_rot, k_rot = apply_rotary_pos_emb(q, k, freqs_cos, freqs_sin)
assert q_rot.shape == q.shape
assert k_rot.shape == k.shape
print(f'freqs_cos shape: {freqs_cos.shape}')
print(f'q_rot shape:     {q_rot.shape}')
print(f'k_rot shape:     {k_rot.shape}')
print('✅ RoPE test passed')

## Part D — Grouped-Query Attention (GQA)

Standard Multi-Head Attention (MHA) uses equal numbers of Q, K, and V heads. **Grouped-Query Attention (GQA)** shares KV heads across multiple query heads, sitting between MHA and Multi-Query Attention (MQA):

| Method | Q Heads | KV Heads | KV Cache Size | Quality |
|---|---|---|---|---|
| MHA (Multi-Head) | 8 | 8 | Full | Best |
| **GQA (Grouped-Query)** | **8** | **4** | **50%** | **Near-MHA** |
| MQA (Multi-Query) | 8 | 1 | 12.5% | Slightly lower |

MiniMind uses **8 query heads** but only **4 KV heads**. Each KV head is shared by a group of 2 query heads (repeat factor = `n_heads / n_kv_heads = 2`). This **halves the KV cache** during inference — critical for serving models where KV cache memory per user is a bottleneck — with minimal quality loss compared to full MHA.

**How attention is computed step-by-step:**
1. **Project** input $x$ into queries ($Q$), keys ($K$), and values ($V$) via learned linear projections — Q has 8 heads, K and V each have 4 heads
2. **Reshape** into per-head views: $Q$ is `[batch, seq, 8, 96]`, $K$/$V$ are `[batch, seq, 4, 96]`
3. **Normalize** Q and K with RMSNorm (QK normalization — prevents attention logits from growing too large, stabilizes training in deep models)
4. **Apply RoPE** to Q and K — inject position information via rotation
5. **Repeat KV** — use `repeat_kv` to replicate each KV head 2× so K and V become `[batch, seq, 8, 96]`, matching Q's head count
6. **Compute attention scores**: $\text{scores} = \frac{Q K^T}{\sqrt{d_k}}$ — the scaled dot-product measures how relevant each key is to each query
7. **Apply causal mask** — set future positions to $-\infty$ so after softmax they become 0 (prevents the model from "seeing" future tokens during training)
8. **Softmax + dropout** — normalize scores to probabilities
9. **Weighted sum**: $\text{output} = \text{scores} \cdot V$ — aggregate values based on attention weights
10. **Output projection** — concatenate heads and project back to `hidden_size`

**Flash Attention:** When available, MiniMind uses PyTorch's `scaled_dot_product_attention` which implements [Flash Attention](https://arxiv.org/abs/2205.14135) — a fused CUDA kernel that computes exact attention in $O(N)$ memory instead of $O(N^2)$ by tiling the computation and avoiding materializing the full attention matrix. This is both **faster** (fewer memory reads) and uses **less GPU memory** (no $N \times N$ attention matrix stored). It is mathematically identical to standard attention — just computed more efficiently.

**Causal masking:** The upper-triangular mask filled with $-\infty$ ensures that position $i$ can only attend to positions $\leq i$. This is what makes the model "autoregressive" — it can only use past context to predict the next token, which is essential for left-to-right language generation.

In [ ]:
# === Grouped-Query Attention ===
def repeat_kv(x, n_rep):
    """Repeat KV heads to match number of query heads."""
    bs, slen, num_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    # ============================================================
    # TODO: Expand KV heads by n_rep times
    #
    # Shape transformation: (bs, slen, num_kv_heads, head_dim)
    #                     → (bs, slen, num_kv_heads * n_rep, head_dim)
    #
    # Steps:
    #   1. Add a new dim: x[:, :, :, None, :]
    #      → shape (bs, slen, num_kv_heads, 1, head_dim)
    #   2. .expand(bs, slen, num_kv_heads, n_rep, head_dim)
    #   3. .reshape(bs, slen, num_kv_heads * n_rep, head_dim)
    # ============================================================

    # YOUR CODE HERE
    pass

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_key_value_heads = config.num_key_value_heads
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = self.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim

        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout
        self.flash = hasattr(F, 'scaled_dot_product_attention') and config.flash_attn

    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        bsz, seq_len, _ = x.shape
        # ============================================================
        # TODO: Implement the attention forward pass
        #
        # Steps:
        #   1. Project to Q, K, V:
        #      xq = self.q_proj(x),  xk = self.k_proj(x),  xv = self.v_proj(x)
        #   2. Reshape to (bsz, seq_len, n_heads, head_dim)
        #   3. Apply QK norms: xq = self.q_norm(xq), xk = self.k_norm(xk)
        #   4. Apply RoPE: xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        #      where cos, sin = position_embeddings
        #   5. Handle KV cache: if past_key_value is not None,
        #      concat past K/V along seq dim; set past_kv if use_cache
        #   6. Transpose to (bsz, n_heads, seq_len, head_dim):
        #      xq = xq.transpose(1,2)
        #      xk = repeat_kv(xk, self.n_rep).transpose(1,2)
        #      xv = repeat_kv(xv, self.n_rep).transpose(1,2)
        #   7. Compute attention:
        #      - If self.flash and seq_len > 1 and no cache:
        #          use F.scaled_dot_product_attention (is_causal=True)
        #      - Else: scores = (xq @ xk.T) / sqrt(head_dim)
        #              add causal mask (upper triangular = -inf)
        #              optionally add attention_mask
        #              output = softmax(scores) @ xv
        #   8. Reshape output: (bsz, seq_len, hidden_size)
        #   9. Apply output projection + residual dropout
        #  10. Return (output, past_kv)
        # ============================================================

        # YOUR CODE HERE
        pass

print('Attention class defined \u2713')


In [ ]:
# === ✅ Milestone 2A: Attention Output Shape ===
config = MiniMindConfig()
attn = Attention(config)
BATCH, SEQ_LEN = 2, 32
x = torch.randn(BATCH, SEQ_LEN, config.hidden_size)

freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=SEQ_LEN)
pos_emb = (freqs_cos, freqs_sin)

with torch.no_grad():
    out, _ = attn(x, pos_emb)

assert out.shape == (BATCH, SEQ_LEN, config.hidden_size), f'Shape: {out.shape}'
print(f'✅ Milestone 2A passed — Attention output shape: {out.shape}')
print(f'   Q heads: {config.num_attention_heads}, KV heads: {config.num_key_value_heads}')
print(f'   GQA repeat factor: {config.num_attention_heads // config.num_key_value_heads}')

## Part E — SwiGLU Feed-Forward Network

Instead of a simple ReLU-MLP, modern transformers use **SwiGLU** — a gated linear unit with the SiLU (Swish) activation:

$$\text{FFN}(x) = W_{\text{down}} \cdot (\text{SiLU}(W_{\text{gate}} \cdot x) \odot W_{\text{up}} \cdot x)$$

where $\text{SiLU}(x) = x \cdot \sigma(x)$ and $\sigma$ is the sigmoid function.

**Evolution of FFN activations in transformers:**
| Architecture | FFN Design | Notes |
|---|---|---|
| Original Transformer | $W_2 \cdot \text{ReLU}(W_1 \cdot x)$ | Simple two-matrix MLP |
| GLU (Gated Linear Unit) | $W_2 \cdot (\sigma(W_{\text{gate}} \cdot x) \odot W_{\text{up}} \cdot x)$ | Adds a gate with sigmoid |
| **SwiGLU** | $W_2 \cdot (\text{SiLU}(W_{\text{gate}} \cdot x) \odot W_{\text{up}} \cdot x)$ | **Replaces sigmoid gate with SiLU** |

**Why gating helps:** The element-wise product ($\odot$) between the gate and up branches lets the network *selectively amplify or suppress* information along each hidden dimension. The gate branch ($\text{SiLU}(W_{\text{gate}} \cdot x)$) acts as a learned filter, while the up branch ($W_{\text{up}} \cdot x$) provides the raw signal. This is more expressive than a single ReLU, which can only zero out dimensions — the gate can smoothly modulate any dimension between 0 and its full value.

**SiLU (Swish) activation:** $\text{SiLU}(x) = x \cdot \sigma(x)$ is a smooth, non-monotonic activation that approximates ReLU for large positive values but allows small negative values through. Unlike ReLU, SiLU has a continuous gradient everywhere, which helps optimization.

**Dimension flow through the FFN:**
1. Input: `[batch, seq, 768]` (hidden_size)
2. Gate projection ($W_{\text{gate}}$): `768 → 2048` (intermediate_size)
3. Up projection ($W_{\text{up}}$): `768 → 2048` (same expansion)
4. Element-wise: `SiLU(gate) × up` → `[batch, seq, 2048]`
5. Down projection ($W_{\text{down}}$): `2048 → 768` (compress back)

Note that SwiGLU uses **three weight matrices** instead of the standard two-matrix MLP. This means for the same `intermediate_size`, SwiGLU has 50% more parameters — but the gating more than compensates in terms of model quality. Production models (LLaMA, Gemma, PaLM) all use SwiGLU or similar gated variants.

In [ ]:
# === SwiGLU Feed-Forward Network ===
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)

    def forward(self, x):
        # ============================================================
        # TODO: Implement SwiGLU feed-forward
        #
        # Formula: down_proj( SiLU(gate_proj(x)) * up_proj(x) )
        #
        # Steps:
        #   1. gate = self.gate_proj(x)         # shape: (B, T, intermediate_size)
        #   2. up   = self.up_proj(x)            # shape: (B, T, intermediate_size)
        #   3. Apply SiLU gate: F.silu(gate) * up
        #   4. Project back down: self.down_proj(...)
        #   5. Return the result
        # ============================================================

        # YOUR CODE HERE
        pass

print('FeedForward (SwiGLU) defined \u2713')


In [ ]:
# === ✅ Milestone 2B: Feed-Forward Shape ===
config = MiniMindConfig()
ffn = FeedForward(config)
x = torch.randn(2, 32, config.hidden_size)

with torch.no_grad():
    out = ffn(x)

assert out.shape == x.shape, f'Shape mismatch: {out.shape}'
print(f'✅ Milestone 2B passed — FFN output shape: {out.shape}')
print(f'   Expansion: {config.hidden_size} → {config.intermediate_size} → {config.hidden_size}')

## Part E½ — Mixture of Experts (MOE)

**Mixture of Experts (MOE)** replaces the standard feed-forward layer with multiple "expert" FFN sub-networks. A learned **gating network** (router) decides which expert(s) each token should be processed by, and the outputs are combined using the gating weights.

**How routing works in detail:**
1. **Gate scores:** A linear layer projects each token's hidden state to $N$ expert scores: $\text{scores} = \text{softmax}(W_g \cdot x)$, where $W_g$ is `[hidden_size, num_experts]`
2. **Top-k selection:** For each token, select the $k$ experts with the highest scores (in MiniMind, $k=1$ by default — each token goes to only one expert)
3. **Normalize weights:** If `norm_topk_prob=True`, the selected weights are renormalized to sum to 1, ensuring consistent output scale
4. **Expert computation:** Each selected expert processes the tokens routed to it — experts are independent `FeedForward` modules
5. **Weighted combination:** Each expert's output is scaled by its gating weight and summed to produce the final output

**Why MOE? Sparse conditional computation:**
- A dense FFN activates all parameters for every token. MOE activates only $k$ out of $N$ experts per token.
- This means MOE can have **many more total parameters** (N experts × FFN parameters each) while keeping the **per-token compute constant** (only k experts run).
- For example, with 4 experts and top-1 routing, MOE has 4× the FFN parameters but the same FLOPs per token as a single FFN.
- This is how models like Mixtral and Switch Transformer achieve high quality with lower inference cost.

**Auxiliary load-balancing loss:** Without any constraint, the router might learn to send all tokens to a single "favorite" expert, making the other experts useless (expert collapse). The auxiliary loss prevents this:

$$\mathcal{L}_{\text{aux}} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot p_i$$

where $f_i$ is the fraction of tokens routed to expert $i$, $p_i$ is the average gating probability assigned to expert $i$, and $\alpha$ is `router_aux_loss_coef`. This loss is minimized when load is evenly distributed, because the product $f_i \cdot p_i$ is minimized when both are uniform ($1/N$ each). The coefficient $\alpha$ (5e-4 in MiniMind) controls how much weight this auxiliary objective has relative to the main language modeling loss.

**Drop-in replacement:** In `MiniMindBlock`, the only change is: `self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)`. The rest of the architecture is identical — MOE is just a different implementation of the FFN sublayer.

In [ ]:
# === Mixture of Experts Feed-Forward ===
class MOEFeedForward(nn.Module):
    """Mixture of Experts: route each token to top-k experts via a learned gate."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.gate = nn.Linear(config.hidden_size, config.num_experts, bias=False)
        self.experts = nn.ModuleList([
            FeedForward(config) for _ in range(config.num_experts)
        ])

    def forward(self, x):
        # ============================================================
        # TODO: Implement Mixture of Experts forward pass
        #
        # Steps:
        #   1. Flatten x to (batch*seq, hidden_dim):
        #      x_flat = x.view(-1, hidden_dim)
        #   2. Compute gating scores:
        #      scores = F.softmax(self.gate(x_flat), dim=-1)
        #   3. Select top-k experts per token:
        #      topk_weight, topk_idx = torch.topk(scores, k=..., dim=-1)
        #   4. Optionally normalize top-k weights:
        #      topk_weight = topk_weight / (topk_weight.sum(dim=-1, keepdim=True) + 1e-20)
        #   5. Dispatch tokens to experts:
        #      for i, expert in enumerate(self.experts):
        #          mask = (topk_idx == i)
        #          if mask.any():
        #              token_idx = mask.any(dim=-1).nonzero().flatten()
        #              weight = topk_weight[mask].view(-1, 1)
        #              y.index_add_(0, token_idx, expert(x_flat[token_idx]) * weight)
        #   6. Compute auxiliary load-balancing loss (self.aux_loss)
        #   7. Reshape output back to (batch, seq, hidden_dim)
        # ============================================================

        # YOUR CODE HERE
        pass

print('MOEFeedForward defined ✓')


In [ ]:
# === ✅ Milestone 2B½: MOE Feed-Forward ===
moe_config = MiniMindConfig()
moe_config.use_moe = True
moe_ff = MOEFeedForward(moe_config)
x = torch.randn(2, 8, moe_config.hidden_size)
out = moe_ff(x)
assert out.shape == x.shape, f'MOE output shape mismatch: {out.shape} != {x.shape}'
print(f'✅ Milestone 2B½ passed — MOE output shape: {out.shape}')
print(f'  Experts: {moe_config.num_experts}, active per token: {moe_config.num_experts_per_tok}')
print(f'  Auxiliary loss: {moe_ff.aux_loss.item():.6f}')


## Part F — Transformer Block

Each transformer block applies two sub-layers with **pre-norm** residual connections:

1. **Pre-norm → Attention → Residual:** `x = x + Attention(RMSNorm(x))`
2. **Pre-norm → FFN → Residual:** `x = x + FFN(RMSNorm(x))`

**Residual connections** (the `+ x` part) are critical for deep networks. They create "shortcut paths" that let gradients flow directly from the loss back to earlier layers without being multiplied through many weight matrices. Without residuals, training networks deeper than ~10 layers becomes extremely difficult due to gradient vanishing.

**Pre-norm vs Post-norm:** The original Transformer (Vaswani et al., 2017) used **post-norm**: `x = LayerNorm(x + Sublayer(x))` — normalize *after* adding the residual. Modern LLMs use **pre-norm**: `x = x + Sublayer(Norm(x))` — normalize *before* the sublayer. Why?

| Aspect | Post-Norm | Pre-Norm |
|---|---|---|
| Gradient flow | Gradients pass through LayerNorm on the residual path | Gradients flow directly through the residual path (unmodified addition) |
| Training stability | Requires careful warmup, prone to instability in deep models | More stable, easier to train without warmup |
| Output scale | Naturally normalized outputs | Requires a final norm layer after the last block |
| Used in | Original Transformer, BERT | LLaMA, GPT, Gemma, MiniMind, most modern LLMs |

The key insight is that with pre-norm, the residual path `x → x + ...` is a pure addition — gradients from the loss reach early layers unmodified through this path. With post-norm, gradients must pass through the LayerNorm on every residual connection, which can attenuate or distort them in deep models.

**Data flow through one MiniMindBlock:**
```
Input x [batch, seq, 768]
   │
   ├──→ RMSNorm ──→ Attention ──→ + residual ──→ hidden_states
   │                                ↑
   └────────────────────────────────┘
   │
   ├──→ RMSNorm ──→ FFN/MOE  ──→ + residual ──→ Output
   │                                ↑
   └────────────────────────────────┘
```

Each block preserves the shape `[batch, seq, hidden_size]` — the same tensor shape flows through all 8 blocks.

In [ ]:
# === Transformer Block ===
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)

    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        # ============================================================
        # TODO: Implement the transformer block with pre-norm residuals
        #
        # Sub-layer 1 — Self-Attention:
        #   residual = hidden_states
        #   norm_x   = self.input_layernorm(hidden_states)
        #   attn_out, present_key_value = self.self_attn(norm_x, position_embeddings,
        #                                                 past_key_value, use_cache, attention_mask)
        #   hidden_states = residual + attn_out       ← residual connection
        #
        # Sub-layer 2 — Feed-Forward:
        #   norm_x       = self.post_attention_layernorm(hidden_states)
        #   hidden_states = hidden_states + self.mlp(norm_x)  ← residual connection
        #
        # Return: (hidden_states, present_key_value)
        # ============================================================

        # YOUR CODE HERE
        pass

print('MiniMindBlock defined \u2713')


In [ ]:
# === Test Transformer Block ===
config = MiniMindConfig()
block = MiniMindBlock(layer_id=0, config=config)
x = torch.randn(2, 32, config.hidden_size)
freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=32)

with torch.no_grad():
    out, _ = block(x, (freqs_cos, freqs_sin))

assert out.shape == x.shape, f'Shape: {out.shape}'
print(f'Block output shape: {out.shape} ✓')

## Part G — Complete MiniMind Model

We stack N transformer blocks, add token embeddings and a language model head to form the complete model:

1. **Token Embedding** — maps token IDs to dense vectors (`vocab_size → hidden_size`). Each of the 6400 tokens in our vocabulary gets a learnable 768-dimensional vector. This is the model's input — converting discrete token IDs into continuous representations the network can process.

2. **N × MiniMindBlock** — the transformer stack. 8 identical blocks (each with attention + FFN), applied sequentially. Each block refines the hidden representations — early layers tend to capture syntax and local patterns, later layers capture semantics and long-range dependencies.

3. **Final RMSNorm** — normalizes the output of the last transformer block. This is necessary because pre-norm leaves the residual path un-normalized (unlike post-norm). Without this final norm, the output scale would grow with depth.

4. **LM Head** — a linear projection from `hidden_size → vocab_size` (768 → 6400) that produces logits over the vocabulary for each position. After softmax, these become next-token probabilities.

**Weight tying:** The embedding matrix and the LM head share the **same weight matrix** (`model.embed_tokens.weight = lm_head.weight`). This is a widely-used technique with two benefits:
- **Fewer parameters** — the embedding matrix is `6400 × 768 = 4.9M` parameters. Without tying, we'd have two copies (one for input, one for output), doubling this cost.
- **Better representations** — the model is forced to use the same vector space for understanding input tokens and predicting output tokens. Tokens that are semantically similar should be nearby in this shared space, which acts as a useful inductive bias.

**How the forward pass works:**
1. `input_ids` `[batch, seq]` → `embed_tokens` → `[batch, seq, 768]`
2. Slice precomputed RoPE frequencies to the current sequence length
3. Pass through all 8 `MiniMindBlock` layers sequentially
4. Apply final `RMSNorm`
5. Project through `lm_head` → logits `[batch, seq, 6400]`

**Loss computation:** When `labels` are provided, the model computes cross-entropy loss with **shifted labels** — `logits[..., :-1, :]` predicts `labels[..., 1:]`. This is because in autoregressive language modeling, the logits at position $i$ should predict the token at position $i+1$. The special label value `-100` is ignored in the loss, which is used later for masking prompt tokens during supervised fine-tuning.

**KV caching for inference:** During generation, we don't want to recompute attention for all previous tokens at each step. The `past_key_values` parameter stores previously computed K and V tensors. At each generation step, only the new token's Q, K, V are computed, and K/V are concatenated with the cache. This makes generation $O(n)$ per step instead of $O(n^2)$.

In [ ]:
# === Complete MiniMind Model ===
class MiniMindModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([
            MiniMindBlock(l, config) for l in range(config.num_hidden_layers)
        ])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        freqs_cos, freqs_sin = precompute_freqs_cis(
            dim=config.head_dim,
            end=config.max_position_embeddings,
            rope_base=config.rope_theta
        )
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False):
        batch_size, seq_length = input_ids.shape
        past_key_values = past_key_values or [None] * len(self.layers)
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0

        # ============================================================
        # TODO: Implement MiniMindModel forward pass
        #
        # Steps:
        #   1. Embed tokens and apply dropout:
        #      hidden_states = self.dropout(self.embed_tokens(input_ids))
        #   2. Slice position embeddings for current sequence:
        #      position_embeddings = (
        #          self.freqs_cos[start_pos : start_pos + seq_length],
        #          self.freqs_sin[start_pos : start_pos + seq_length]
        #      )
        #   3. Pass through each layer, collecting KV cache:
        #      presents = []
        #      for layer, past_kv in zip(self.layers, past_key_values):
        #          hidden_states, present = layer(hidden_states, position_embeddings,
        #                                         past_key_value=past_kv, use_cache=use_cache,
        #                                         attention_mask=attention_mask)
        #          presents.append(present)
        #   4. Apply final norm: hidden_states = self.norm(hidden_states)
        #   5. Compute aux_loss from MOE layers:
        #      aux_loss = sum([l.mlp.aux_loss for l in self.layers if isinstance(l.mlp, MOEFeedForward)], ...)
        #   6. Return (hidden_states, presents, aux_loss)
        # ============================================================

        # YOUR CODE HERE
        pass


class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model = MiniMindModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        # Weight tying: share embedding and output projection weights
        self.model.embed_tokens.weight = self.lm_head.weight

    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, use_cache=False):
        # ============================================================
        # TODO: Implement the causal LM forward pass
        #
        # Steps:
        #   1. Call self.model(...) to get hidden_states, past_key_values, and aux_loss
        #   2. Project to vocabulary: logits = self.lm_head(hidden_states)
        #   3. If labels is not None, compute cross-entropy loss:
        #      - Shift: shift_logits = logits[..., :-1, :].contiguous()
        #               shift_labels = labels[..., 1:].contiguous()
        #      - Loss: F.cross_entropy(shift_logits.view(-1, vocab_size),
        #                              shift_labels.view(-1), ignore_index=-100)
        #   4. Return {'loss': loss, 'logits': logits, 'past_key_values': past_key_values, 'aux_loss': aux_loss}
        # ============================================================

        # YOUR CODE HERE
        pass

print('MiniMindForCausalLM defined \u2713')


In [ ]:
# === ✅ Milestone 2C: Complete Model & Parameter Count ===
config = MiniMindConfig()
model = MiniMindForCausalLM(config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Milestone 2C passed — complete model built')
print(f'   Total parameters:     {total_params:>12,}')
print(f'   Trainable parameters: {trainable_params:>12,}')
print(f'   Model size (MB):      {total_params * 4 / 1024 / 1024:>12.1f}')

# Quick forward pass test
input_ids = torch.randint(0, config.vocab_size, (2, 32))
with torch.no_grad():
    outputs = model(input_ids)

assert outputs['logits'].shape == (2, 32, config.vocab_size), f'Logits shape: {outputs["logits"].shape}'
print(f'   Logits shape:         {outputs["logits"].shape}')

# Test with labels (loss computation)
labels = torch.randint(0, config.vocab_size, (2, 32))
with torch.no_grad():
    outputs = model(input_ids, labels=labels)

assert outputs['loss'] is not None, 'Loss should not be None when labels provided'
print(f'   Loss (random init):   {outputs["loss"].item():.4f}')
print(f'   Expected ~ln({config.vocab_size})={math.log(config.vocab_size):.2f} for random model')

In [ ]:
# === Parameter Breakdown by Layer ===
print('Parameter breakdown:')
print('-' * 55)
for name, param in model.named_parameters():
    if 'layers.0.' in name or 'embed' in name or 'lm_head' in name or 'norm.weight' == name.split('.')[-1] and 'layers' not in name:
        if 'layers.0.' in name:
            layer_name = name.replace('model.layers.0.', 'block.')
            count = param.numel()
            print(f'  {layer_name:45s} {count:>10,}  {list(param.shape)}')
        elif 'embed' in name:
            print(f'  {"embed_tokens":45s} {param.numel():>10,}  {list(param.shape)}')

# Per-block total
block_params = sum(p.numel() for n, p in model.named_parameters() if 'layers.0.' in n)
print(f'\n  Per-block total: {block_params:,} × {config.num_hidden_layers} layers = {block_params * config.num_hidden_layers:,}')
embed_params = model.model.embed_tokens.weight.numel()
print(f'  Embedding (tied with lm_head): {embed_params:,}')
print(f'  Final norm: {model.model.norm.weight.numel():,}')

# Summary

| Component | Key Concept |
|---|---|
| **MiniMindConfig** | Hyperparameters: 768 hidden, 8 layers, 8 Q / 4 KV heads |
| **RMSNorm** | Normalize by RMS (no mean subtraction), learnable scale |
| **RoPE** | Encode position via rotation matrices in 2D subspaces |
| **GQA Attention** | Share KV heads across Q head groups (4 KV → 8 Q) |
| **SwiGLU FFN** | Gated linear unit: down(SiLU(gate(x)) × up(x)) |
| **MOE FFN** | Route tokens to top-k of N experts via learned gate |
| **MiniMindBlock** | Pre-norm → Attention → Residual → Pre-norm → FFN/MOE → Residual |
| **MiniMindForCausalLM** | Full model with weight-tied embedding & LM head |

**Milestones completed:** 2A (Attention shape), 2B (FFN shape), 2B½ (MOE shape), 2C (complete model)

### Next Notebook
In Notebook 3, we build the **data pipeline** — loading and preprocessing text data for pretraining and supervised fine-tuning, with proper label masking.
